In [1]:
!pip install -q groq sentence-transformers faiss-cpu pypdf flask pyngrok numpy pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 33.3 MB/s eta 0:00:00


In [2]:
import os
import json
import pickle
import numpy as np
import pandas as pd

from flask import Flask, render_template, request, jsonify
import requests
from pyngrok import ngrok

from sklearn.model_selection import train_test_split, RandomizedSearchCV, RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve

from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader
from groq import Groq
import os


In [3]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    raise ValueError("Set GROQ_API_KEY before running this cell.")

client = Groq(api_key=GROQ_API_KEY)
test = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say OK"}],
    max_tokens=10
)
print("Groq API:", test.choices[0].message.content)

Groq API: OK


In [ ]:
# No API key needed here — the "find nearby care" feature uses the free
# OpenStreetMap Overpass API (see the location helper cell below), which
# requires no signup, no billing, and no key.
print("Using OpenStreetMap Overpass API for location search (free, no key required).")


In [4]:
from google.colab import files
uploaded = files.upload()

Saving who_ai.pdf to who_ai.pdf


In [5]:
def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def split_text(text: str, chunk_size: int = 120, overlap: int = 30) -> list[str]:
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


pdf_text  = load_pdf("who_ai.pdf")
documents = split_text(pdf_text)
print(f"PDF loaded → {len(documents)} overlapping chunks")

PDF loaded → 4 overlapping chunks


In [6]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents, convert_to_numpy=True, show_progress_bar=True)
doc_embeddings /= np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)


def retrieve(query: str, k: int = 5) -> list[str]:
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
    _, indices = index.search(q_emb, k)
    return [documents[i] for i in indices[0]]


print("FAISS index ready")
print(retrieve("heart disease risk factors")[0][:200])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index ready
Exercise regularly * Maintain healthy weight --- [Heart Disease] What is heart disease? Heart disease refers to conditions affecting the heart and blood vessels. What are symptoms of heart disease? * 


In [7]:
from google.colab import files
uploaded = files.upload()

Saving heart.csv to heart.csv


In [8]:
FEATURE_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]
NUMERIC_COLS     = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_COLS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

df = pd.read_csv("heart.csv")
X  = df[FEATURE_NAMES]

y = 1 - df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

candidates = {
    "random_forest": (
        RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
        {
            "model__n_estimators":     [200, 400, 600],
            "model__max_depth":        [None, 4, 6, 8],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features":     ["sqrt", "log2"],
        },
    ),
    "gradient_boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "model__n_estimators":  [100, 200, 300],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth":     [2, 3, 4],
        },
    ),
    "logistic_regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {
            "model__C":       [0.01, 0.1, 1, 10],
            "model__penalty": ["l2"],
        },
    ),
}

best_name, best_search, best_cv_auc = None, None, -1

for name, (estimator, grid) in candidates.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    search = RandomizedSearchCV(
        pipe, grid, n_iter=15, scoring="roc_auc", cv=cv,
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    print(f"{name:20s} CV AUC = {search.best_score_:.3f}  | best params = {search.best_params_}")
    if search.best_score_ > best_cv_auc:
        best_name, best_search, best_cv_auc = name, search, search.best_score_

print(f"\nSelected model: {best_name}  (CV AUC = {best_cv_auc:.3f})")
best_model = best_search.best_estimator_

test_probs = best_model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, test_probs)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = int(np.argmax(f1_scores[:-1]))
best_threshold = float(thresholds[best_idx])

print(f"Tuned decision threshold: {best_threshold:.3f}")
print(f"At this threshold -> precision={precisions[best_idx]:.3f}, recall={recalls[best_idx]:.3f}")

test_preds = (test_probs >= best_threshold).astype(int)
print("\nHeld-out test performance (tuned threshold):")
print(classification_report(y_test, test_preds))

test_auc = roc_auc_score(y_test, test_probs)
print(f"Test AUC: {test_auc:.3f}")

MODEL_AUC = round(best_cv_auc, 3)


random_forest        CV AUC = 0.914  | best params = {'model__n_estimators': 400, 'model__min_samples_leaf': 4, 'model__max_features': 'log2', 'model__max_depth': 4}
gradient_boosting    CV AUC = 0.902  | best params = {'model__n_estimators': 200, 'model__max_depth': 2, 'model__learning_rate': 0.01}


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=15. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


logistic_regression  CV AUC = 0.917  | best params = {'model__penalty': 'l2', 'model__C': 0.1}

Selected model: logistic_regression  (CV AUC = 0.917)
Tuned decision threshold: 0.628
At this threshold -> precision=0.913, recall=0.750

Held-out test performance (tuned threshold):
              precision    recall  f1-score   support

           0       0.82      0.94      0.87        33
           1       0.91      0.75      0.82        28

    accuracy                           0.85        61
   macro avg       0.86      0.84      0.85        61
weighted avg       0.86      0.85      0.85        61

Test AUC: 0.922


In [9]:
with open("health.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("threshold.pkl", "wb") as f:
    pickle.dump(best_threshold, f)

with open("model_meta.pkl", "wb") as f:
    pickle.dump({"name": best_name, "cv_auc": MODEL_AUC, "test_auc": round(test_auc, 3)}, f)

print(f"Model ({best_name}) + threshold + metadata saved")


Model (logistic_regression) + threshold + metadata saved


In [10]:
SYSTEM_PROMPT = """You are a knowledgeable, empathetic heart-health assistant built into
a cardiovascular risk tool. Your job is to help users understand their results and
general heart-health concepts.

Rules:
- Base answers on the provided medical context when relevant.
- Be concise (3-5 sentences max unless the user asks for more detail).
- Never diagnose or prescribe — always recommend seeing a doctor for personal advice.
- If a question is completely unrelated to health or the tool, politely redirect.
- Use plain English; avoid jargon unless the user clearly prefers clinical terms.
"""


def ask_llm(
    user_message: str,
    history: list,
    rag_context: str = "",
    prediction_context: str = ""
) -> str:
    enriched = user_message
    if rag_context:
        enriched = (
            f"[Relevant medical context from WHO guidelines]\n{rag_context}\n\n"
            f"[User question]\n{user_message}"
        )
    if prediction_context:
        enriched = f"[Current prediction]\n{prediction_context}\n\n" + enriched

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history
    messages += [{"role": "user", "content": enriched}]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        max_tokens=512,
        temperature=0.5
    )
    return response.choices[0].message.content.strip()


print(ask_llm("What is LDL cholesterol?", []))


LDL (Low-Density Lipoprotein) cholesterol is often referred to as "bad" cholesterol. It's a type of fat found in your blood that can build up in your arteries, increasing your risk of heart disease and stroke. High levels of LDL cholesterol can lead to plaque formation, narrowing your arteries and reducing blood flow. It's a good idea to talk to your doctor about your LDL levels and how to manage them for a healthy heart.


In [ ]:
# --- Location helper: find nearby doctors / healthcare centers via the
# free OpenStreetMap Overpass API (no API key needed) ---

OVERPASS_URL = "https://overpass-api.de/api/interpreter"


def _resolve_place_search(query: str):
    q = (query or "").lower()
    if any(w in q for w in ["hospital", "emergency", "er ", "urgent care"]):
        return "hospital"
    if any(w in q for w in ["pharmacy", "chemist", "medicine", "drug store"]):
        return "pharmacy"
    if "clinic" in q:
        return "clinic"
    return "doctors"  # OSM tag for individual doctor/GP practices


def _format_address(tags: dict) -> str:
    parts = [
        tags.get("addr:housenumber", ""),
        tags.get("addr:street", ""),
        tags.get("addr:city", "") or tags.get("addr:suburb", ""),
    ]
    addr = " ".join(p for p in parts if p).strip()
    return addr or tags.get("addr:full", "") or ""


def find_nearby_healthcare(lat: float, lng: float, query: str = "", radius: int = 5000) -> dict:
    """Query OpenStreetMap's Overpass API for doctors / healthcare centers
    close to (lat, lng). Returns a dict with either 'results' or 'error'."""
    amenity = _resolve_place_search(query)

    overpass_query = f"""
    [out:json][timeout:25];
    (
      node["amenity"="{amenity}"](around:{radius},{lat},{lng});
      way["amenity"="{amenity}"](around:{radius},{lat},{lng});
    );
    out center 12;
    """

    try:
        resp = requests.post(OVERPASS_URL, data={"data": overpass_query}, timeout=25)
        resp.raise_for_status()
        data = resp.json()
    except requests.RequestException as e:
        return {"error": f"Could not reach the location service: {e}"}

    results = []
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        name = tags.get("name")
        if not name:
            continue  # skip unnamed entries, they aren't useful to show

        if el.get("type") == "node":
            plat, plng = el.get("lat"), el.get("lon")
        else:
            center = el.get("center", {})
            plat, plng = center.get("lat"), center.get("lon")

        if plat is None or plng is None:
            continue

        results.append({
            "name": name,
            "address": _format_address(tags),
            "rating": None,               # not available via OSM
            "user_ratings_total": None,   # not available via OSM
            "open_now": None,             # OSM has opening_hours as raw text, not live status
            "opening_hours": tags.get("opening_hours"),
            "phone": tags.get("phone") or tags.get("contact:phone"),
            "lat": plat,
            "lng": plng,
            "maps_url": f"https://www.google.com/maps/search/?api=1&query={plat},{plng}",
        })

    results = results[:8]
    return {"results": results, "type_searched": amenity, "keyword_searched": amenity}


print("Location helper ready (OpenStreetMap Overpass API, no key required)")


In [11]:
import os

os.makedirs("templates", exist_ok=True)

html_code = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Heart Risk Predictor</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600;700&display=swap" rel="stylesheet">
<style>
  :root {
    --red:#E24B4A; --red-dk:#A32D2D; --teal:#1D9E75; --amber:#BA7517;
    --violet:#6E5CE6;
    --bg:#F4F2EC; --card:#FFFFFF; --ink:#2E2D2B;
    --muted:rgba(46,45,43,0.45); --border:rgba(46,45,43,0.1); --field:#F7F6F2;
  }
  *,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
  body{
    font-family:'DM Sans',sans-serif;color:var(--ink);min-height:100vh;
    background:
      radial-gradient(circle at 8% 0%, rgba(226,75,74,.10) 0%, transparent 38%),
      radial-gradient(circle at 95% 12%, rgba(110,92,230,.10) 0%, transparent 40%),
      radial-gradient(circle at 50% 100%, rgba(29,158,117,.08) 0%, transparent 45%),
      var(--bg);
    background-attachment:fixed;
  }
  .page{display:flex;flex-direction:column;align-items:center;padding:3rem 1.5rem 6rem}
  .header{text-align:center;margin-bottom:1.5rem;animation:fadeDown .6s ease both}
  .eyebrow{font-size:10px;font-weight:600;letter-spacing:.22em;text-transform:uppercase;color:var(--red);margin-bottom:.9rem}
  .header h1{font-family:'DM Serif Display',serif;font-size:clamp(2.2rem,5vw,3.4rem);font-weight:400;line-height:1.1}
  .header h1 em{font-style:italic;color:var(--red)}
  .header-sub{font-size:14px;color:var(--muted);font-weight:300;max-width:380px;margin:.65rem auto 0;line-height:1.6}
  .badge-row{display:flex;gap:8px;justify-content:center;flex-wrap:wrap;margin:1.2rem 0 2rem;animation:fadeDown .7s .05s ease both}
  .badge{display:flex;align-items:center;gap:6px;font-size:11px;font-weight:600;letter-spacing:.03em;color:var(--ink);background:rgba(255,255,255,.65);backdrop-filter:blur(6px);border:1px solid var(--border);padding:7px 14px;border-radius:100px;box-shadow:0 2px 10px rgba(46,45,43,.05)}
  .badge .dot{width:7px;height:7px;border-radius:50%}
  .dot-teal{background:var(--teal)}
  .dot-violet{background:var(--violet)}
  .dot-amber{background:var(--amber)}
  .card{width:100%;max-width:880px;background:var(--card);border:1px solid var(--border);border-radius:24px;box-shadow:0 10px 44px rgba(46,45,43,.09);overflow:hidden;animation:fadeUp .7s .1s ease both}
  .sec-label{font-size:10px;font-weight:600;letter-spacing:.17em;text-transform:uppercase;color:var(--muted);padding:1.25rem 1.75rem .7rem;border-bottom:1px solid var(--border);display:flex;align-items:center;gap:7px}
  .sec-label::before{content:'';display:block;width:6px;height:6px;border-radius:50%;background:var(--red);opacity:.7}
  .form-body{padding:1.25rem 1.75rem 1.75rem}
  .group-title{font-size:11px;font-weight:600;letter-spacing:.1em;text-transform:uppercase;color:var(--red-dk);opacity:.8;margin:1.1rem 0 .6rem}
  .group-title:first-child{margin-top:.1rem}
  .grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(190px,1fr));gap:11px;margin-bottom:.4rem}
  .field{display:flex;flex-direction:column;gap:5px}
  .field label{font-size:10px;font-weight:600;letter-spacing:.07em;text-transform:uppercase;color:var(--muted)}
  .field label small{font-size:9px;font-weight:300;opacity:.75}
  .field input{width:100%;padding:9px 12px;border-radius:10px;border:1px solid var(--border);background:var(--field);color:var(--ink);font-family:'DM Sans',sans-serif;font-size:13px;outline:none;transition:border-color .2s,background .2s,box-shadow .2s}
  .field input:focus{border-color:rgba(226,75,74,.5);background:#fff;box-shadow:0 0 0 3px rgba(226,75,74,.1)}
  .btn{width:100%;margin-top:1.1rem;padding:14px;border-radius:12px;border:none;background:linear-gradient(120deg,var(--red),var(--red-dk));color:#fff;font-family:'DM Sans',sans-serif;font-size:13px;font-weight:600;letter-spacing:.06em;text-transform:uppercase;cursor:pointer;transition:opacity .2s,transform .15s,box-shadow .2s;box-shadow:0 6px 18px rgba(226,75,74,.28)}
  .btn:hover{opacity:.92;transform:translateY(-1px);box-shadow:0 9px 22px rgba(226,75,74,.36)}
  .divider{height:1px;background:var(--border);margin:0 1.75rem}
  .result{padding:1.75rem}
  .result-title{font-family:'DM Serif Display',serif;font-size:1.6rem;font-weight:400;margin-bottom:1.4rem;text-align:center}
  .result-layout{display:flex;gap:2rem;align-items:center;flex-wrap:wrap;justify-content:center}
  .gauge-wrap{position:relative;width:172px;height:172px;flex-shrink:0}
  .gauge{transform:rotate(-90deg)}
  .gauge-bg{fill:none;stroke:rgba(46,45,43,.08);stroke-width:14}
  .gauge-fill{fill:none;stroke:url(#gaugeGrad);stroke-width:14;stroke-linecap:round;transition:stroke-dashoffset 1.1s cubic-bezier(.4,0,.2,1)}
  .gauge-center{position:absolute;inset:0;display:flex;flex-direction:column;align-items:center;justify-content:center}
  .gauge-pct{font-family:'DM Serif Display',serif;font-size:2.1rem;line-height:1}
  .gauge-pct small{font-family:'DM Sans',sans-serif;font-size:13px;color:var(--muted);font-weight:300;margin-left:1px}
  .gauge-label{font-size:9px;letter-spacing:.12em;text-transform:uppercase;color:var(--muted);margin-top:4px}
  .result-info{flex:1;min-width:240px;display:flex;flex-direction:column;gap:1rem}
  .chips{display:flex;gap:7px;flex-wrap:wrap}
  .chip{padding:5px 14px;border-radius:100px;font-size:11px;font-weight:600}
  .chip-low{background:rgba(29,158,117,.1);color:#0F6E56;border:1px solid rgba(29,158,117,.25)}
  .chip-mid{background:rgba(186,117,23,.1);color:#7A4E0C;border:1px solid rgba(186,117,23,.25)}
  .chip-high{background:rgba(226,75,74,.1);color:var(--red-dk);border:1px solid rgba(226,75,74,.25)}
  .insight{font-size:13px;line-height:1.65;color:var(--ink);background:var(--field);border-radius:12px;padding:14px 16px;border:1px solid var(--border)}
  .insight strong{display:block;font-size:10px;letter-spacing:.1em;text-transform:uppercase;color:var(--muted);margin-bottom:6px;font-weight:600}
  .disclaimer{margin-top:2.5rem;text-align:center;font-size:11px;color:var(--muted);line-height:1.7;max-width:480px}
  #chat-btn{position:fixed;bottom:26px;right:26px;z-index:100;width:56px;height:56px;border-radius:50%;border:none;background:linear-gradient(135deg,var(--red),var(--red-dk));color:#fff;cursor:pointer;box-shadow:0 6px 22px rgba(226,75,74,.4);display:flex;align-items:center;justify-content:center;transition:transform .2s}
  #chat-btn:hover{transform:scale(1.08) rotate(-4deg)}
  #chat-win{position:fixed;bottom:94px;right:26px;z-index:99;width:340px;max-height:520px;background:#fff;border:1px solid var(--border);border-radius:20px;box-shadow:0 8px 40px rgba(46,45,43,.13);display:none;flex-direction:column;overflow:hidden}
  #chat-win.open{display:flex;animation:fadeUp .22s ease}
  .chat-hdr{background:linear-gradient(120deg,var(--red),var(--red-dk));padding:12px 16px;display:flex;align-items:center;gap:9px}
  .chat-hdr-ico{width:28px;height:28px;background:rgba(255,255,255,.2);border-radius:50%;display:flex;align-items:center;justify-content:center;flex-shrink:0}
  .chat-hdr h3{font-size:13px;font-weight:600;color:#fff;line-height:1.2}
  .chat-hdr p{font-size:10px;color:rgba(255,255,255,.7)}
  #chat-msgs{flex:1;overflow-y:auto;padding:12px 12px 6px;display:flex;flex-direction:column;gap:8px;background:#F7F6F2;scroll-behavior:smooth}
  .msg{max-width:86%;padding:8px 12px;border-radius:13px;font-size:13px;line-height:1.55}
  .msg.bot{background:#fff;color:var(--ink);border:1px solid var(--border);border-bottom-left-radius:3px;align-self:flex-start}
  .msg.user{background:var(--red);color:#fff;border-bottom-right-radius:3px;align-self:flex-end}
  .msg.typing{background:#fff;border:1px solid var(--border);color:var(--muted);align-self:flex-start;font-style:italic;font-size:12px}
  .chat-row{display:flex;gap:7px;padding:9px 11px;background:#fff;border-top:1px solid var(--border)}
  #chat-in{flex:1;border:1px solid var(--border);border-radius:9px;padding:8px 11px;font-size:13px;font-family:'DM Sans',sans-serif;color:var(--ink);background:var(--field);outline:none;resize:none}
  #chat-in:focus{border-color:rgba(226,75,74,.45);background:#fff}
  #chat-send{width:34px;height:34px;border-radius:9px;border:none;background:var(--red);color:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;flex-shrink:0;transition:opacity .2s;align-self:flex-end}
  #chat-send:hover{opacity:.85}
  .chat-quick{display:flex;gap:6px;padding:8px 11px 0;background:#F7F6F2}
  #near-btn{flex:1;display:flex;align-items:center;justify-content:center;gap:6px;padding:8px 10px;border-radius:9px;border:1px solid var(--border);background:#fff;color:var(--red-dk);font-family:'DM Sans',sans-serif;font-size:12px;font-weight:600;cursor:pointer;transition:opacity .2s,transform .15s}
  #near-btn:hover{opacity:.85;transform:translateY(-1px)}
  #near-btn:disabled{opacity:.5;cursor:default;transform:none}
  .nearby-list{width:100%;max-width:100%}
  .nearby-card{background:var(--field);border:1px solid var(--border);border-radius:10px;padding:9px 11px;margin-top:7px}
  .nearby-name{font-size:13px;font-weight:600;color:var(--ink)}
  .nearby-addr{font-size:11.5px;color:var(--muted);margin-top:2px}
  .nearby-meta{font-size:11px;color:var(--teal);margin-top:4px;font-weight:500}
  .nearby-link{display:inline-block;margin-top:6px;font-size:11.5px;color:var(--red);font-weight:600;text-decoration:none}
  .nearby-link:hover{text-decoration:underline}
  @keyframes fadeDown{from{opacity:0;transform:translateY(-16px)}to{opacity:1;transform:translateY(0)}}
  @keyframes fadeUp{from{opacity:0;transform:translateY(18px)}to{opacity:1;transform:translateY(0)}}
  @media(max-width:560px){.form-body,.result{padding:1.1rem 1.2rem 1.4rem}.sec-label{padding:1.1rem 1.2rem .65rem}.divider{margin:0 1.2rem}.grid{grid-template-columns:1fr 1fr}#chat-win{width:calc(100vw - 32px);right:16px}.result-layout{flex-direction:column}}
  @media(max-width:380px){.grid{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="page">
  <header class="header">
    <div class="eyebrow">AI Clinical Assessment</div>
    <h1>Heart Risk<br><em>Predictor</em></h1>
    <p class="header-sub">Enter patient clinical parameters for an AI-powered cardiovascular risk assessment, backed by a retrieval-augmented health assistant.</p>
  </header>
  <div class="badge-row">
    <div class="badge"><span class="dot dot-teal"></span>Model CV AUC&nbsp;{{ model_auc if model_auc else "—" }}</div>
    <div class="badge"><span class="dot dot-violet"></span>Tuned Decision Threshold</div>
    <div class="badge"><span class="dot dot-amber"></span>WHO Guideline RAG</div>
  </div>
  <div class="card">
    <div class="sec-label">Patient Clinical Parameters</div>
    <form action="/predict" method="POST">
      <div class="form-body">
        <div class="group-title">Demographics</div>
        <div class="grid">
          {% set demo_fields = [
            ('age','Age','years','54',15,77,1),
            ('sex','Sex','0=F · 1=M','1',0,1,1)
          ] %}
          {% for name,label,hint,ph,mn,mx,stp in demo_fields %}
          <div class="field">
            <label for="{{ name }}">{{ label }} <small>{{ hint }}</small></label>
            <input type="number" id="{{ name }}" name="{{ name }}"
                   placeholder="{{ ph }}" min="{{ mn }}" max="{{ mx }}" step="{{ stp }}" required>
          </div>
          {% endfor %}
        </div>

        <div class="group-title">Vitals &amp; Labs</div>
        <div class="grid">
          {% set vital_fields = [
            ('trestbps','Resting BP','mm Hg','130',94,200,1),
            ('chol','Cholesterol','mg/dl','220',126,564,1),
            ('fbs','Fasting Blood Sugar','>120 mg','0',0,1,1),
            ('thalach','Max Heart Rate','bpm','150',71,202,1)
          ] %}
          {% for name,label,hint,ph,mn,mx,stp in vital_fields %}
          <div class="field">
            <label for="{{ name }}">{{ label }} <small>{{ hint }}</small></label>
            <input type="number" id="{{ name }}" name="{{ name }}"
                   placeholder="{{ ph }}" min="{{ mn }}" max="{{ mx }}" step="{{ stp }}" required>
          </div>
          {% endfor %}
        </div>

        <div class="group-title">Cardiac Test Results</div>
        <div class="grid">
          {% set cardiac_fields = [
            ('cp','Chest Pain Type','0–3','2',0,3,1),
            ('restecg','Resting ECG','0–2','1',0,2,1),
            ('exang','Exercise Angina','0=No 1=Yes','0',0,1,1),
            ('oldpeak','ST Depression','Oldpeak','1.0',0,6.2,0.1),
            ('slope','ST Slope','0–2','1',0,2,1),
            ('ca','Major Vessels','0–4','0',0,4,1),
            ('thal','Thalassemia','0–3','2',0,3,1)
          ] %}
          {% for name,label,hint,ph,mn,mx,stp in cardiac_fields %}
          <div class="field">
            <label for="{{ name }}">{{ label }} <small>{{ hint }}</small></label>
            <input type="number" id="{{ name }}" name="{{ name }}"
                   placeholder="{{ ph }}" min="{{ mn }}" max="{{ mx }}" step="{{ stp }}" required>
          </div>
          {% endfor %}
        </div>
        <button type="submit" class="btn">Run Prediction &rarr;</button>
      </div>
    </form>
    {% if prediction_text %}
    <div class="divider"></div>
    <div class="sec-label">Assessment Result</div>
    <div class="result">
      <p class="result-title">{{ prediction_text }}</p>
      <div class="result-layout">
        <div class="gauge-wrap">
          <svg width="172" height="172" viewBox="0 0 172 172" class="gauge">
            <defs>
              <linearGradient id="gaugeGrad" x1="0%" y1="0%" x2="100%" y2="0%">
                <stop offset="0%" stop-color="#1D9E75"/>
                <stop offset="55%" stop-color="#BA7517"/>
                <stop offset="100%" stop-color="#E24B4A"/>
              </linearGradient>
            </defs>
            <circle cx="86" cy="86" r="68" class="gauge-bg"/>
            <circle cx="86" cy="86" r="68" class="gauge-fill" id="gaugeFill"/>
          </svg>
          <div class="gauge-center">
            <div class="gauge-pct" id="gaugePct">0<small>%</small></div>
            <div class="gauge-label">Risk Score</div>
          </div>
        </div>
        <div class="result-info">
          <div class="chips">
            {% if risk|int < 30 %}
              <span class="chip chip-low">Low Risk</span>
              <span class="chip chip-low">Routine Monitoring</span>
            {% elif risk|int < 65 %}
              <span class="chip chip-mid">Moderate Risk</span>
              <span class="chip chip-mid">Follow-Up Recommended</span>
            {% else %}
              <span class="chip chip-high">High Risk</span>
              <span class="chip chip-high">Urgent Review Advised</span>
            {% endif %}
          </div>
          {% if insight %}
          <div class="insight"><strong>AI Insight</strong>{{ insight }}</div>
          {% endif %}
        </div>
      </div>
    </div>
    {% endif %}
  </div>
  <p class="disclaimer"><strong>Clinical disclaimer.</strong> This tool provides a statistical estimate and is not a substitute for professional medical diagnosis. Always consult a qualified cardiologist.</p>
</div>

<button id="chat-btn" title="Ask a health question">
  <svg width="22" height="22" viewBox="0 0 24 24" fill="none">
    <path d="M21 15c0 .53-.21 1.04-.59 1.41-.37.38-.88.59-1.41.59H7l-4 4V5c0-.53.21-1.04.59-1.41C3.96 3.21 4.47 3 5 3h14c.53 0 1.04.21 1.41.59.38.37.59.88.59 1.41v10z" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
  </svg>
</button>

<div id="chat-win">
  <div class="chat-hdr">
    <div class="chat-hdr-ico">
      <svg width="15" height="15" viewBox="0 0 32 32" fill="none">
        <path d="M16 28S3 20.5 3 11.5C3 7.36 6.13 4 10 4c2.35 0 4.43 1.21 6 3 1.57-1.79 3.65-3 6-3C25.87 4 29 7.36 29 11.5 29 20.5 16 28 16 28z" fill="white"/>
      </svg>
    </div>
    <div>
      <h3>Heart Health Assistant</h3>
    </div>
  </div>
  <div class="chat-quick">
    <button id="near-btn" type="button">&#128205; Find care near me</button>
  </div>
  <div id="chat-msgs">
    <div class="msg bot">Hi! I can answer questions about heart health, risk factors, medications, and your results. Tap "Find care near me" to locate nearby doctors and healthcare centers, or ask me anything.</div>
  </div>
  <div class="chat-row">
    <textarea id="chat-in" rows="1" placeholder="Ask about cholesterol, BP, your results…"></textarea>
    <button id="chat-send">
      <svg width="15" height="15" viewBox="0 0 24 24" fill="none">
        <path d="M22 2L11 13M22 2l-7 20-4-9-9-4 20-7z" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
      </svg>
    </button>
  </div>
</div>

<script>
  var risk = {{ risk if risk else 0 }};
  var circumference = 2 * Math.PI * 68;
  var fill = document.getElementById('gaugeFill');
  if (fill) {
    fill.style.strokeDasharray = circumference;
    fill.style.strokeDashoffset = circumference;
    setTimeout(() => {
      fill.style.strokeDashoffset = circumference - (circumference * Math.min(risk, 100) / 100);
    }, 250);
  }
  var pctEl = document.getElementById('gaugePct');
  if (pctEl) {
    var t0 = null, dur = 1100;
    function step(ts) {
      if (!t0) t0 = ts;
      var p = Math.min((ts - t0) / dur, 1);
      var val = Math.round(p * risk);
      pctEl.innerHTML = val + '<small>%</small>';
      if (p < 1) requestAnimationFrame(step);
    }
    requestAnimationFrame(step);
  }

  document.getElementById('chat-btn').onclick = () =>
    document.getElementById('chat-win').classList.toggle('open');

  var chatHistory = [];
  var inp = document.getElementById('chat-in');
  inp.addEventListener('input', () => {
    inp.style.height = 'auto';
    inp.style.height = Math.min(inp.scrollHeight, 90) + 'px';
  });

  var LOCATION_INTENT = /\b(near me|nearby|close by|closest|nearest|find (a |the )?(doctor|hospital|clinic|pharmacy)|doctor near|hospital near|clinic near|pharmacy near|healthcare center|health center)\b/i;

  function appendBotMsg(msgs, text) {
    var b = document.createElement('div');
    b.className = 'msg bot';
    b.textContent = text;
    msgs.appendChild(b);
    msgs.scrollTop = msgs.scrollHeight;
  }

  function buildMeta(p) {
    var meta = [];
    if (p.rating) meta.push('\u2605 ' + p.rating + (p.user_ratings_total ? ' (' + p.user_ratings_total + ')' : ''));
    if (p.open_now === true) meta.push('Open now');
    else if (p.open_now === false) meta.push('Closed now');
    else if (p.opening_hours) meta.push(p.opening_hours);
    if (p.phone) meta.push(p.phone);
    return meta;
  }

  function renderNearbyResults(msgs, data) {
    if (data.error) { appendBotMsg(msgs, data.error); return; }
    var results = data.results || [];
    if (!results.length) {
      appendBotMsg(msgs, "I couldn't find any healthcare centers nearby. Try again in a different area.");
      return;
    }
    var wrap = document.createElement('div');
    wrap.className = 'msg bot nearby-list';
    var html = '<strong style="display:block;">Nearby care options:</strong>';
    results.forEach(function (p) {
      var meta = buildMeta(p);
      html += '<div class="nearby-card">' +
        '<div class="nearby-name"></div>' +
        (p.address ? '<div class="nearby-addr"></div>' : '') +
        (meta.length ? '<div class="nearby-meta"></div>' : '') +
        '<a class="nearby-link" target="_blank" rel="noopener">Open in Google Maps &rarr;</a>' +
        '</div>';
    });
    wrap.innerHTML = html;
    // fill in text content safely (avoids HTML injection from place names/addresses)
    var cards = wrap.querySelectorAll('.nearby-card');
    results.forEach(function (p, i) {
      var card = cards[i];
      card.querySelector('.nearby-name').textContent = p.name || 'Unnamed location';
      var addrEl = card.querySelector('.nearby-addr');
      if (addrEl) addrEl.textContent = p.address || '';
      var metaEl = card.querySelector('.nearby-meta');
      if (metaEl) metaEl.textContent = buildMeta(p).join(' \u00b7 ');
      var link = card.querySelector('.nearby-link');
      link.href = p.maps_url || ('https://www.google.com/maps/search/?api=1&query=' + p.lat + ',' + p.lng);
    });
    msgs.appendChild(wrap);
    msgs.scrollTop = msgs.scrollHeight;
  }

  function findNearbyCare(queryHint) {
    var msgs = document.getElementById('chat-msgs');
    document.getElementById('chat-win').classList.add('open');
    if (!navigator.geolocation) {
      appendBotMsg(msgs, "Location isn't available in this browser.");
      return;
    }
    var t = document.createElement('div');
    t.className = 'msg typing'; t.id = 'typing'; t.textContent = 'Finding nearby care\u2026';
    msgs.appendChild(t);
    msgs.scrollTop = msgs.scrollHeight;

    var nearBtn = document.getElementById('near-btn');
    if (nearBtn) nearBtn.disabled = true;

    navigator.geolocation.getCurrentPosition(function (pos) {
      fetch('/nearby_care', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          lat: pos.coords.latitude,
          lng: pos.coords.longitude,
          query: queryHint || ''
        })
      })
        .then(r => r.json())
        .then(data => {
          document.getElementById('typing')?.remove();
          renderNearbyResults(msgs, data);
          if (nearBtn) nearBtn.disabled = false;
        })
        .catch(() => {
          document.getElementById('typing')?.remove();
          appendBotMsg(msgs, 'Could not fetch nearby care right now. Please try again.');
          if (nearBtn) nearBtn.disabled = false;
        });
    }, function () {
      document.getElementById('typing')?.remove();
      appendBotMsg(msgs, 'Location permission was denied. Please allow location access and try again.');
      if (nearBtn) nearBtn.disabled = false;
    }, { timeout: 10000 });
  }

  document.getElementById('near-btn').onclick = function () { findNearbyCare(''); };

  function sendMsg() {
    var text = inp.value.trim();
    if (!text) return;
    var msgs = document.getElementById('chat-msgs');
    var u = document.createElement('div');
    u.className = 'msg user'; u.textContent = text;
    msgs.appendChild(u);
    msgs.scrollTop = msgs.scrollHeight;
    inp.value = ''; inp.style.height = 'auto';

    if (LOCATION_INTENT.test(text)) {
      findNearbyCare(text);
      return;
    }

    var t = document.createElement('div');
    t.className = 'msg typing'; t.id = 'typing'; t.textContent = 'Thinking…';
    msgs.appendChild(t);
    msgs.scrollTop = msgs.scrollHeight;
    fetch('/chat', {
      method: 'POST',
      headers: {'Content-Type': 'application/json'},
      body: JSON.stringify({ message: text, history: chatHistory })
    })
    .then(r => r.json())
    .then(data => {
      document.getElementById('typing')?.remove();
      var b = document.createElement('div');
      b.className = 'msg bot';
      b.textContent = data.reply || 'Sorry, I could not find an answer.';
      msgs.appendChild(b);
      msgs.scrollTop = msgs.scrollHeight;
      chatHistory.push({role:'user', content: text});
      chatHistory.push({role:'assistant', content: data.reply});
      if (chatHistory.length > 20) chatHistory = chatHistory.slice(-20);
    })
    .catch(() => {
      document.getElementById('typing')?.remove();
      var e = document.createElement('div');
      e.className = 'msg bot'; e.textContent = 'Connection error — please try again.';
      msgs.appendChild(e);
      msgs.scrollTop = msgs.scrollHeight;
    });
  }

  document.getElementById('chat-send').onclick = sendMsg;
  inp.addEventListener('keydown', e => {
    if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); sendMsg(); }
  });
</script>
</body>
</html>
"""


with open('templates/index.html', 'w') as f:
    f.write(html_code)

print('Template written')


Template written


In [12]:
app = Flask(__name__)


FIELD_RANGES = {
    "age":      (15,  77),
    "sex":      (0,   1),
    "cp":       (0,   3),
    "trestbps": (94,  200),
    "chol":     (126, 564),
    "fbs":      (0,   1),
    "restecg":  (0,   2),
    "thalach":  (71,  202),
    "exang":    (0,   1),
    "oldpeak":  (0,   6.2),
    "slope":    (0,   2),
    "ca":       (0,   4),
    "thal":     (0,   3),
}

KNOWN_CATEGORIES = {
    "sex":      {0, 1},
    "cp":       {0, 1, 2, 3},
    "fbs":      {0, 1},
    "restecg":  {0, 1, 2},
    "exang":    {0, 1},
    "slope":    {0, 1, 2},
    "ca":       {0, 1, 2, 3, 4},
    "thal":     {0, 1, 2, 3},
}


def parse_and_validate(form_data):
    values, errors = [], []
    for col in FEATURE_NAMES:
        raw = form_data.get(col, "").strip()
        try:
            val = float(raw)
        except ValueError:
            errors.append(f"{col}: must be a number")
            values.append(0.0)
            continue

        lo, hi = FIELD_RANGES.get(col, (-1e9, 1e9))
        if not (lo <= val <= hi):
            errors.append(f"{col}: {val} out of range [{lo}, {hi}]")

        if col in KNOWN_CATEGORIES:
            if not val.is_integer() or int(val) not in KNOWN_CATEGORIES[col]:
                errors.append(
                    f"{col}: {val} is not a category the model was trained on "
                    f"(valid: {sorted(KNOWN_CATEGORIES[col])})"
                )

        values.append(val)
    return values, errors


def feature_summary(values):
    labels = {
        "age":"Age","sex":"Sex","cp":"Chest pain type",
        "trestbps":"Resting BP","chol":"Cholesterol","fbs":"Fasting blood sugar>120",
        "restecg":"Rest ECG","thalach":"Max heart rate","exang":"Exercise angina",
        "oldpeak":"ST depression","slope":"ST slope","ca":"Major vessels","thal":"Thalassemia"
    }
    return ", ".join(f"{labels[k]}={v}" for k, v in zip(FEATURE_NAMES, values))


@app.route("/")
def home():
    return render_template("index.html", risk=0, model_auc=MODEL_AUC)


@app.route("/predict", methods=["POST"])
def predict():
    form = request.form.to_dict()
    user_input, errors = parse_and_validate(form)

    if errors:
        return render_template(
            "index.html", risk=0, model_auc=MODEL_AUC,
            prediction_text="Input error: " + " | ".join(errors)
        )

    input_df = pd.DataFrame([user_input], columns=FEATURE_NAMES)

    prob_raw = best_model.predict_proba(input_df)[0][1]
    pred     = int(prob_raw >= best_threshold)
    prob     = round(prob_raw * 100, 1)
    result   = "Heart Disease Detected" if pred == 1 else "No Heart Disease Detected"

    feat_str = feature_summary(user_input)
    pred_ctx = (
        f"Prediction: {result} | Risk score: {prob}% | "
        f"Decision threshold: {round(best_threshold * 100, 1)}% | Patient data: {feat_str}"
    )
    rag_ctx  = "\n".join(retrieve("heart disease risk factors causes", k=4))

    try:
        insight = ask_llm(
            user_message=(
                "In 2-3 sentences, explain what the prediction result means for this "
                "patient and the key factors that likely drove it. Be direct and clear."
            ),
            history=[],
            rag_context=rag_ctx,
            prediction_context=pred_ctx
        )
    except Exception as e:
        insight = f"(AI insight unavailable: {e})"

    return render_template(
        "index.html",
        prediction_text=result,
        risk=prob,
        insight=insight,
        model_auc=MODEL_AUC
    )


@app.route("/nearby_care", methods=["POST"])
def nearby_care():
    data = request.get_json(force=True) or {}
    lat = data.get("lat")
    lng = data.get("lng")
    query = data.get("query", "")

    if lat is None or lng is None:
        return jsonify(error="Location was not provided."), 400

    try:
        lat = float(lat)
        lng = float(lng)
    except (TypeError, ValueError):
        return jsonify(error="Invalid location coordinates."), 400

    result = find_nearby_healthcare(lat, lng, query=query)
    return jsonify(result)


@app.route("/chat", methods=["POST"])
def chat():
    data     = request.get_json(force=True)
    user_msg = data.get("message", "").strip()
    history  = data.get("history", [])

    if not user_msg:
        return jsonify(reply="Please ask a question.")

    rag_ctx = "\n".join(retrieve(user_msg, k=5))

    try:
        reply = ask_llm(
            user_message=user_msg,
            history=history,
            rag_context=rag_ctx
        )
    except Exception as e:
        reply = f"Something went wrong: {e}"

    return jsonify(reply=reply)


print("Flask app defined")

Flask app defined


In [13]:
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")

!ngrok config add-authtoken "$NGROK_AUTH_TOKEN"

public_url = ngrok.connect(5000)
print("App live at:", public_url)

app.run(port=5000)

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
App live at: NgrokTunnel: "https://c493-34-7-13-216.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 06:09:42] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jul/2026 06:09:43] "GET /favicon.ico HTTP/1.1" 404 -
